# Closed-loop episode inspector

Scrub one closed-loop rollout and **draw** the vocabulary trajectories that pass the scores you check.

The scene below is validation index **540** (`ce0be5afd6b03891`), the turn from the diversity set. Weights default to `checkpoints/hydra_4s_sde_triple_par_metfix/best.pt`, which has the imitation head, the EPO head, and the metric heads (`nc`, `dac`, `ddc`, `tlc`, `ttc`, `lk`, `hc`, `progress`). The car replans every 0.1 s. Neighbors follow the log. The episode stops at the first collision or off-road step. On this checkpoint that happens at about 1.7 s, off the roundabout. `im_pert` has no metric head, so predicted `nc` / `ttc` / … stay empty there; ground truth still draws.

**Ground truth** is the rule scorer in the executed ego frame at that timestep: a trajectory is kept when its label is at least the threshold. Imitation ground truth is closeness to the **log** future (`exp(−ADE / scale)`). EPO ground truth is rule EPDMS.

**Predicted** uses the network at that same pose. Metric boxes are probabilities. Imitation and EPO are that head’s softmax divided by its peak, so the threshold is a fraction of the best mode (the best mode is 1).

Several checkboxes together keep the trajectories that pass **all** of them. Color is the weakest checked score. Cyan is the plan the closed-loop policy actually drove (dashed when it fails the filter). Black is the log, red is where the car has already driven.


In [8]:
from pathlib import Path

from womd_il.viz.episode_inspect import capture_episode, episode_summary

def find_root() -> Path:
    here = Path.cwd().resolve()
    for cand in (here, here.parent, Path("/home/florentin/repos/myprojects/waymo_proj")):
        if (cand / "src" / "womd_il").exists():
            return cand
    return here

ROOT = find_root()
CKPT = ROOT / "checkpoints/hydra_4s_sde_triple_par_metfix/best.pt"
H5 = ROOT / "data/womd/packed/val.h5"
SCENARIO = 480
HORIZON = 80
H_EXEC = 1
MODE_SELECT = "epo"  # avg | geomean | epo | imitation | pdms
EARLY_STOP = True  # stop at the first collision or off-road step
DEVICE = "cuda"
CACHE = CKPT.parent / "inspect" / f"i{SCENARIO}_{MODE_SELECT}.npz"

episode, sample = capture_episode(
    CKPT,
    H5,
    SCENARIO,
    device=DEVICE,
    horizon=HORIZON,
    h_exec=H_EXEC,
    mode_select=MODE_SELECT,
    neighbors="log",
    early_stop_on_fail=EARLY_STOP,
    cache_path=CACHE,
)
print(episode_summary(episode))
print(f"cache {CACHE}")


Hydra decoder: 4096 trajectory proposals (T=40) stages=(4096,) tokenizer=mlp cross_only=True dual_cross=True triple_cross=True triple_parallel=True ego_on_query=True main=mix policy_head=epo imi_scale=1 epo_scale=1 epo_ent=0.05 epo_tau=1 epo_ent_min_k=256 metric_stopgrad=False metric_loss_scale=1 metric_class_weighted=True metric_rate=count metric_pos_weight_cap=None nms_radius=0 keep_winner=False vocab_self_attn=0 metric_mlp=3x256


8f66d6790b964ef2  idx=480  2 frames (0.1s)  replans=1  collision=False  offroad=True  K=4096  predicted metrics=yes  select=epo
cache /home/florentin/repos/myprojects/waymo_proj/outputs/inspect/i480_hydra_4s_sde_triple_par_metfix_epo.npz


In [9]:
import ipywidgets as widgets
from IPython.display import display

from womd_il.viz.episode_inspect import CHANNELS, render_episode_png

TIPS = {
    "imitation": "Log closeness (ground truth) or imitation-head mass (predicted)",
    "epo": "Rule EPDMS (ground truth) or EPO-head mass (predicted)",
    "nc": "No at-fault collision over 4 s",
    "dac": "Stays on the drivable surface",
    "ddc": "Driving-direction compliance",
    "tlc": "Traffic-light compliance",
    "ttc": "Time-to-collision clear",
    "lk": "Lane keeping",
    "hc": "History comfort",
    "progress": "Progress along the log future",
}

source = widgets.ToggleButtons(
    options=[("ground truth", "gt"), ("predicted", "pred")],
    value="gt",
    description="source",
)
slider = widgets.IntSlider(
    value=0,
    min=0,
    max=max(episode.n_frames - 1, 0),
    step=1,
    description="time",
    continuous_update=False,
    layout=widgets.Layout(width="640px"),
    readout=False,
)
time_readout = widgets.HTML()
threshold = widgets.FloatSlider(
    value=0.5, min=0.0, max=1.0, step=0.05, description="threshold",
    continuous_update=False, readout_format=".2f",
)
ade_scale = widgets.FloatSlider(
    value=2.0, min=0.5, max=8.0, step=0.5, description="ADE scale",
    continuous_update=False, readout_format=".1f",
)
max_draw = widgets.IntSlider(
    value=800, min=50, max=4096, step=50, description="max lines",
    continuous_update=False,
)
radius = widgets.FloatSlider(
    value=40.0, min=15.0, max=80.0, step=5.0, description="radius m",
    continuous_update=False,
)
checks = {
    name: widgets.Checkbox(value=(name == "nc"), description=name, indent=False, tooltip=TIPS[name])
    for name in CHANNELS
}
show_log = widgets.Checkbox(value=True, description="log", indent=False)
show_executed = widgets.Checkbox(value=True, description="executed", indent=False)
show_pick = widgets.Checkbox(value=True, description="executed plan", indent=False)
show_agents = widgets.Checkbox(value=True, description="agents", indent=False)
image = widgets.Image(format="png", layout=widgets.Layout(width="760px"))
status = widgets.HTML()

def refresh(_=None):
    sec = slider.value * 0.1
    time_readout.value = f"{sec:.1f}s &nbsp; step {slider.value + 1}/{episode.n_frames}"
    try:
        png, html = render_episode_png(
            episode,
            sample,
            frame=int(slider.value),
            source=source.value,
            selected=[name for name, box in checks.items() if box.value],
            threshold=float(threshold.value),
            ade_scale=float(ade_scale.value),
            max_draw=int(max_draw.value),
            radius=float(radius.value),
            show_pick=bool(show_pick.value),
            show_log=bool(show_log.value),
            show_executed=bool(show_executed.value),
            show_agents=bool(show_agents.value),
        )
    except Exception as exc:
        status.value = f"<pre>{type(exc).__name__}: {exc}</pre>"
        return
    image.value = png
    status.value = html

for widget in (
    source, slider, threshold, ade_scale, max_draw, radius,
    show_log, show_executed, show_pick, show_agents, *checks.values(),
):
    widget.observe(refresh, names="value")

grid = widgets.GridBox(
    list(checks.values()),
    layout=widgets.Layout(grid_template_columns="repeat(5, 130px)", grid_gap="4px 8px"),
)
display(
    widgets.VBox([
        widgets.HBox([slider, time_readout]),
        widgets.HBox([source, threshold]),
        widgets.HBox([ade_scale, max_draw, radius]),
        grid,
        widgets.HBox([show_log, show_executed, show_pick, show_agents]),
        image,
        status,
    ])
)
refresh()
